In [3]:
# import os
# import json
# import geopandas as gpd
# import pandas as pd
# from shapely.geometry import Point

# PARCEL_DIR = "./data_files/parcels/Parcel_Digest_2025"
# AIRBNB_JSON = "./data_files/in_process_data/locations.js"   # adjust if needed

def load_parcel_geometries(parcel_dir):
    """
    Load parcel geometries from GeoJSON (preferred), SHP, or KML.
    Ensures CRS is EPSG:4326 before computing centroids.
    """
    candidates = [
        "Parcel_Digest_2025.geojson",   # always prefer this
        "Parcel_Digest_2025.kml",
        "Parcel_Digest_2025.shp"
    ]

    gdf = None
    for fname in candidates:
        path = os.path.join(parcel_dir, fname)
        if os.path.exists(path):
            print(f"Loading parcel geometry from: {path}")
            gdf = gpd.read_file(path)
            break

    if gdf is None:
        raise FileNotFoundError("No parcel geometry file found.")

    # Ensure CRS is EPSG:4326
    if gdf.crs is None:
        raise ValueError("Parcel file has no CRS defined.")

    if gdf.crs.to_epsg() != 4326:
        print(f"Reprojecting from {gdf.crs} → EPSG:4326")
        gdf = gdf.to_crs("EPSG:4326")

    # Force correct address column
    if "PropAddress_Full" in gdf.columns:
        addr_col = "PropAddress_Full"
    else:
        raise ValueError("PropAddress_Full not found in parcel data.")

    # Compute centroids AFTER CRS fix
    gdf["centroid"] = gdf.geometry.centroid

    return gdf, addr_col




def load_airbnb_json(path):
    """
    Loads your JS file containing the locations array.
    Assumes the file contains: const locations = [ ... ];
    """
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    # Extract JSON array
    start = text.find("[")
    end = text.rfind("]") + 1
    json_text = text[start:end]

    locations = json.loads(json_text)
    return locations


def match_airbnb_to_parcels(locations, parcels_gdf, addr_col):
    """
    For each Airbnb listing, find nearest parcel centroid.
    """
    # Build GeoDataFrame of Airbnb points
    airbnb_points = gpd.GeoDataFrame(
        locations,
        geometry=[Point(loc["longitude"], loc["latitude"]) for loc in locations],
        crs="EPSG:4326"
    )

    # Ensure parcels are in same CRS
    parcels_gdf = parcels_gdf.to_crs("EPSG:4326")

    # Build centroid GeoDataFrame
    centroids = gpd.GeoDataFrame(
        parcels_gdf[[addr_col]],
        geometry=parcels_gdf["centroid"],
        crs="EPSG:4326"
    )

    # Spatial nearest join
    joined = gpd.sjoin_nearest(
        airbnb_points,
        centroids,
        how="left",
        distance_col="dist_meters"
    )

    # Inject address into original dicts
    updated_locations = []
    for loc, row in zip(locations, joined.itertuples()):
        loc["searched_full_address"] = getattr(row, addr_col)
        loc["approx_distance_meters"] = getattr(row, "dist_meters")
        updated_locations.append(loc)

    return updated_locations


def write_updated_json(path, updated_locations):
    """
    Writes updated JSON back into the JS file format.
    """
    js_text = "const locations = " + json.dumps(updated_locations, indent=2) + ";"
    with open(path, "w", encoding="utf-8") as f:
        f.write(js_text)
    print("Updated JSON written to:", path)


if __name__ == "__main__":
    parcels_gdf, addr_col = load_parcel_geometries(PARCEL_DIR)
    locations = load_airbnb_json(AIRBNB_JSON)
    updated = match_airbnb_to_parcels(locations, parcels_gdf, addr_col)
    write_updated_json(AIRBNB_JSON, updated)


Loading parcel geometry from: ./data_files/parcels/Parcel_Digest_2025\Parcel_Digest_2025.geojson


C:\Users\micha\AppData\Local\Temp\ipykernel_11692\2963432170.py:47: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid"] = gdf.geometry.centroid
C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Updated JSON written to: ./data_files/in_process_data/locations.js
